In [1]:
from datasets import load_dataset
import pandas as pd

# 下载数据集（test split）
ds = load_dataset("TIGER-Lab/MMLU-Pro", split="test")

# 转为 DataFrame
df = ds.to_pandas()
print(f"In total {len(df)} 条记录，字段：{list(df.columns)}")
df.head(5)

/home/snt/projects_lujun/LLMJudgeTempCausal/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In total 12032 条记录，字段：['question_id', 'question', 'options', 'answer', 'answer_index', 'cot_content', 'category', 'src']


,question_id,question,options,answer,answer_index,cot_content,category,src
0,70,"Typical advertising regulatory bodies suggest,...","[Safe practices, Fear, Jealousy, Trivial, Unsa...",I,8,,business,ori_mmlu-business_ethics
1,71,Managers are entrusted to run the company in t...,"[Shareholders, Diligence, Self-interest, Share...",F,5,,business,ori_mmlu-business_ethics
2,72,There are two main issues associated with ____...,"[Down, Autonomy, Remuneration, Benefit, Down, ...",J,9,,business,ori_mmlu-business_ethics
3,73,_______ locate morality beyond the sphere of r...,"[Ethical egoism, Ethics of duty, Postmodern et...",C,2,,business,ori_mmlu-business_ethics
4,74,Some of key differences between Islamic finan...,"[Interest, Certain, Assured, Both tangible and...",G,6,,business,ori_mmlu-business_ethics


In [2]:
# 按学科统计题目数量
print("各学科题目数：")
display(df["category"].value_counts().rename_axis("category").reset_index(name="count"))

# 看一道题的完整内容
sample = df.iloc[0]
print(f"\n题目：{sample['question']}")
print(f"选项：{sample['options']}")
print(f"正确答案：{sample['answer']} (index {sample['answer_index']})")

各学科题目数：


,category,count
0,math,1351
1,physics,1299
2,chemistry,1132
3,law,1101
4,engineering,969
5,other,924
6,economics,844
7,health,818
8,psychology,798
9,business,789



题目：Typical advertising regulatory bodies suggest, for example that adverts must not: encourage _________, cause unnecessary ________ or _____, and must not cause _______ offence.
选项：['Safe practices, Fear, Jealousy, Trivial'
 'Unsafe practices, Distress, Joy, Trivial'
 'Safe practices, Wants, Jealousy, Trivial'
 'Safe practices, Distress, Fear, Trivial'
 'Unsafe practices, Wants, Jealousy, Serious'
 'Safe practices, Distress, Jealousy, Serious'
 'Safe practices, Wants, Fear, Serious'
 'Unsafe practices, Wants, Fear, Trivial'
 'Unsafe practices, Distress, Fear, Serious']
正确答案：I (index 8)


In [3]:
target_n = 150

# 按 category 比例计算每类应抽多少条
category_counts = df["category"].value_counts().sort_index()
raw_counts = category_counts / len(df) * target_n
sample_counts = raw_counts.astype(int)

# 把余下的名额按小数部分从大到小补齐，确保总数正好是 150
remainder = target_n - sample_counts.sum()
if remainder > 0:
    extra_categories = (raw_counts - sample_counts).sort_values(ascending=False).index[:remainder]
    sample_counts.loc[extra_categories] += 1

# 分类别抽样
sampled_parts = []
for category, n in sample_counts.items():
    part = df[df["category"] == category].sample(n=int(n), random_state=42)
    sampled_parts.append(part)

sampled_df = pd.concat(sampled_parts).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"抽样后共 {len(sampled_df)} 条")
print("\n抽样后的 category 分布：")
display(sampled_df["category"].value_counts().rename_axis("category").reset_index(name="count"))

sampled_df.head()

抽样后共 150 条

抽样后的 category 分布：


,category,count
0,math,17
1,physics,16
2,law,14
3,chemistry,14
4,engineering,12
5,other,11
6,economics,11
7,business,10
8,health,10
9,psychology,10


,question_id,question,options,answer,answer_index,cot_content,category,src
0,4715,This question refers to the following informat...,[Logan believes that Indians need to find stre...,F,5,,history,ori_mmlu-high_school_us_history
1,208,You are asked to determine the price of a Euro...,"[16.4, 9.7, 11.9, 15.6, 13.1, 14.2, 7.8, 8.5, ...",C,2,,business,theoremQA-Finance
2,11245,When was the major shift by Greek philosopher...,"[Late Sixth Century BCE, Late Second Century B...",A,0,,philosophy,ori_mmlu-world_religions
3,1286,Which of the following criticisms of Llewellyn...,[There is no distinction between the two forms...,C,2,,law,ori_mmlu-jurisprudence
4,1200,Is the recognition of foreign judgments subjec...,[Foreign judgments are enforced on the basis o...,D,3,,law,ori_mmlu-international_law


In [ ]:
sampled_df_mmlu